# Validation Croisée Temporelle : Modèle XGBoost (Walk-Forward)

Ce notebook présente une version nettoyée et robuste de l'évaluation du modèle **XGBoost** pour la prévision des arrivées touristiques.

Contrairement à un modèle classique qui peut souffrir de **Data Leakage** si les données sont mélangées, nous appliquons ici une méthode de **Walk-Forward Validation** (via `TimeSeriesSplit`).

## Objectifs :
1. Charger et préparer les données.
2. Différencier les séries pour garantir la stationnarité.
3. Entraîner récursivement le XGBoost sur une fenêtre en expansion.
4. Évaluer les performances réelles sur des données strictement futures à chaque itération.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import os

# Chemins
data_dir = '../data'
df_x_train = pd.read_csv(os.path.join(data_dir, 'separted', 'X_train.csv'))
df_y_train = pd.read_csv(os.path.join(data_dir, 'separted', 'y_train.csv'))
df_x_test = pd.read_csv(os.path.join(data_dir, 'separted', 'X_test.csv'))
df_y_test = pd.read_csv(os.path.join(data_dir, 'separted', 'y_test.csv'))

# Nettoyage
columns_to_drop = ['GDP_Construction_lag1']
df_x_train = df_x_train.drop(columns=columns_to_drop, errors='ignore')
df_x_test = df_x_test.drop(columns=columns_to_drop, errors='ignore')

# Imputation et Scaling global (pas de leakage ici car il s'agit juste de features exogènes standard)
imputer = SimpleImputer(strategy='mean')
df_x_train_imputed = imputer.fit_transform(df_x_train)
df_x_test_imputed = imputer.transform(df_x_test)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(df_x_train_imputed)
x_test_scaled = scaler.transform(df_x_test_imputed)

# Stationnarité : Différenciation de la cible
y_train_diff = df_y_train.diff().dropna()
y_test_diff = df_y_test.diff().dropna()

X_train_diff = x_train_scaled[1:]
X_test_diff = x_test_scaled[1:]

X_all_diff = np.vstack((X_train_diff, X_test_diff))
y_all_diff = pd.concat([y_train_diff, y_test_diff]).values.flatten()


## Entraînement Walk-Forward
Nous utilisons `TimeSeriesSplit` pour créer des fenêtres d'entraînement successives.

In [ ]:
xgb_model = xgb.XGBRegressor(
    learning_rate=0.1, max_depth=4, n_estimators=150, 
    reg_alpha=0, reg_lambda=10, subsample=0.7, random_state=42
)

tscv = TimeSeriesSplit(n_splits=len(X_test_diff), test_size=1)
y_pred_diff_wf = []
test_start_idx = len(X_train_diff)

print("Début de l'entraînement récursif...")
for train_index, test_index in tscv.split(X_all_diff):
    if test_index[0] < test_start_idx:
        continue
    
    # Le modèle ne voit que le passé à l'instant t
    X_train_wf, X_test_wf = X_all_diff[train_index], X_all_diff[test_index]
    y_train_wf, y_test_wf = y_all_diff[train_index], y_all_diff[test_index]
    
    xgb_model.fit(X_train_wf, y_train_wf)
    pred = xgb_model.predict(X_test_wf)
    y_pred_diff_wf.append(pred[0])

# Reconstruire les prédictions finales (inverser la différence)
valeurs_ref = df_y_test.iloc[:-1].values.flatten()
y_pred_finales = valeurs_ref + np.array(y_pred_diff_wf)
y_true_aligned = df_y_test.iloc[1:].values.flatten()

metrics = {
    'R2': r2_score(y_true_aligned, y_pred_finales),
    'RMSE': np.sqrt(mean_squared_error(y_true_aligned, y_pred_finales)),
    'MAE': mean_absolute_error(y_true_aligned, y_pred_finales),
    'MAPE': mean_absolute_percentage_error(y_true_aligned, y_pred_finales)
}
print(f"\nMétriques Finales XGBoost Walk-Forward :\n{metrics}")

plt.figure(figsize=(10,5))
plt.plot(y_true_aligned, label='Réel', color='black')
plt.plot(y_pred_finales, label='Prédiction Walk-Forward', color='red', linestyle='--')
plt.title('XGBoost - Réel vs Prédiction')
plt.legend()
plt.grid()
plt.show()
